# Pokémon Gen 2 OU — Three reproducible submissions

This notebook reproduces the three **model variants** from one training run: observed offense, full boosted baseline, and unanimous-expert consensus. Only the organizer's official competition files are used as input data.

The three original Kaggle-selected CSVs still require a one-to-one prediction comparison before an exact reproduction of those uploads can be claimed. Exploratory cross-validation results do not guarantee leaderboard performance.

## 1. Clone the repository and install dependencies

This notebook uses the same source commit as the successful Kaggle execution: `0f006fd126277c6bf03ea13f63402cb8364035e6`. The pinned checkout avoids changing model behavior if the repository's default branch is updated. Internet access is needed for Git and package installation, not as a modeling input.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/fds-pokemon")
REPO_URL = "https://github.com/lucamartone/fds-pokemon.git"
SOURCE_COMMIT = "0f006fd126277c6bf03ea13f63402cb8364035e6"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / ".git").is_dir():
    raise ValueError(f"Not a Git repository: {REPO_DIR}")
else:
    origin = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True
    ).strip()
    if origin != REPO_URL:
        raise ValueError(f"Unexpected repository remote: {origin}")

subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", SOURCE_COMMIT], check=True)
COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert COMMIT == SOURCE_COMMIT, f"Unexpected source commit: {COMMIT}"
print("Repository commit:", COMMIT)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True
)


## 2. Locate official competition files and generate all three submissions

The original `run.py` entrypoint is executed once. It fits the boosted baseline, then the roster, hybrid-resource, and symmetric-orientation experts, generating all three outputs without using labels from the test data.

In [ ]:
train_files = list(Path("/kaggle/input").rglob("train.jsonl"))
if len(train_files) != 1:
    raise FileNotFoundError(f"Expected exactly one train.jsonl; found {len(train_files)}")
DATA_DIR = train_files[0].parent
for name in ("test.jsonl", "type_chart_gen2.csv", "sample_submission.csv"):
    if not (DATA_DIR / name).is_file():
        raise FileNotFoundError(DATA_DIR / name)
OUTPUT_DIR = Path("/kaggle/working/submissions")
subprocess.run([
    sys.executable, str(REPO_DIR / "run.py"),
    "--data", str(DATA_DIR), "--output-dir", str(OUTPUT_DIR)
], cwd=REPO_DIR, check=True)
print("Output directory:", OUTPUT_DIR, "Commit:", COMMIT)


In [ ]:
import hashlib
import pandas as pd

EXPECTED_FILES = (
    "submission_1_offense.csv",
    "submission_2_full.csv",
    "submission_3_consensus.csv",
)
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")

def inspect_submission(filename):
    assert filename in EXPECTED_FILES, filename
    path = OUTPUT_DIR / filename
    submission = pd.read_csv(path)
    assert submission.shape == sample.shape, filename
    assert submission.columns.equals(sample.columns), filename
    assert submission.battle_id.equals(sample.battle_id), filename
    assert submission.battle_id.is_unique, filename
    assert submission.player_won.isin((0, 1)).all(), filename
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{filename}: {len(submission)} rows; sha256={digest}")
    display(submission.head(3))
    return submission


## 3. Submission 1 — Observed offense

**File:** `submission_1_offense.csv`

The offense ensemble blends the original LightGBM/XGBoost/CatBoost baseline with a compact logistic regression, active-species categorical features and CatBoost features derived from damaging moves observed in the first 30 turns. This is the intermediate `candidate_7001` blend in the source, not a claim about its Kaggle score.

In [ ]:
submission_1 = inspect_submission("submission_1_offense.csv")

## 4. Submission 2 — Full boosted baseline

**File:** `submission_2_full.csv`

The full baseline extends the offense model with observed effective speed, move priority and utility moves, including recovery, status and setup. Its fixed blending recipe is labeled `candidate_7010` in the source. Training labels are used only through the official training data.

In [ ]:
submission_2 = inspect_submission("submission_2_full.csv")

## 5. Submission 3 — Unanimous-expert consensus

**File:** `submission_3_consensus.csv`

This model starts from the full boosted baseline and changes a low-confidence decision only when **all three independent experts** agree on the opposite class: revealed-roster, hybrid-resource and mirrored-orientation CatBoost experts. The baseline probability must be within 0.10 of 0.5. No earlier Kaggle submissions are used as model inputs.

In [ ]:
submission_3 = inspect_submission("submission_3_consensus.csv")

## 6. Reproducibility and final delivery

All three CSV variants are written to `/kaggle/working/submissions/`. Their checks verify row count, columns, ordered unique `battle_id` values, binary predictions and a SHA-256 checksum.

For a claim of **exact reproduction of the three selected Kaggle entries**, compare each generated CSV against the corresponding original uploaded CSV. The score listing alone does not identify that mapping. No model source was changed in preparing this notebook.